# 01 – Data Acquisition & Cleaning

This notebook pulls data from two APIs and saves two clean CSV files to `data/raw/`.
**The rest of the group only needs these CSV files to get there analysis. They don't need to run this notebook or have an API key.**

| File | One row per | Sorted by | Columns |
|---|---|---|---|
| `census_places.csv` | U.S. city/town | population, largest first | geoid, city, state, population, median_income, pct_bike_commute, pct_no_vehicle |
| `citybike_networks_us.csv` | U.S. bike-share network | number of stations, most first | network_id, name, city, state_abbr, n_stations, total_docks, pct_missing_capacity |

Load them with:
```python
census = pd.read_csv('data/raw/census_places.csv', dtype={'geoid': str})   # keeps geoid's leading zeros
bikes = pd.read_csv('data/raw/citybike_networks_us.csv')
```

## 1. Setup: imports, API key, output folder

The Census key is **never typed into this notebook**:
- **Colab:** in the Secrets panel (key icon), add a secret named `CENSUS_API_KEY`.
- **Local:** put `CENSUS_API_KEY=...` in `.env`, which is gitignored.

Don't print the key, because cell output gets saved into the notebook(safe coding practices).

In [17]:
import os
import time
from pathlib import Path
import requests
import pandas as pd

try:
    from google.colab import userdata              # Colab: key lives in the Secrets panel
    CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')
except ImportError:
    from dotenv import load_dotenv                 # Local: key lives in .env (gitignored)
    load_dotenv()
    CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')
assert CENSUS_API_KEY, 'No Census API key found - see README Setup'

# repo's data/raw when run from notebooks/, else ./data/raw (Colab - download the files before the session ends)
DATA_DIR = Path('../data/raw') if Path('../data').exists() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. CityBikes: U.S. networks

`/v2/networks` lists every bike-share network in the world. We **keep only the U.S. ones**.
CityBikes city names look like `"Chicago, IL"`, so we split them into `city` and `state_abbr`.

In [18]:

# STEP 1: Get the list of every bike-share network, then keep only U.S. ones


# The /v2/networks endpoint lists every bike-share system in the world (no API key needed)
response_bikes = requests.get('https://api.citybik.es/v2/networks', timeout=30)
response_bikes.raise_for_status()      # stop here with a clear error if the request failed (e.g. 404 or 500)

# Each network has a nested "location" field (city, country, coordinates).
# json_normalize flattens it into columns like 'location.city' and 'location.country'
all_networks = pd.json_normalize(response_bikes.json()['networks'])

networks_df = (all_networks[all_networks['location.country'] == 'US']   # keep only U.S. networks
    .rename(columns={'id': 'network_id',                                 # clearer name, matches stations table
                     'location.city': 'location'})                       # e.g. "New York, NY"
    [['network_id', 'name', 'location']])                                # keep only the columns we need

# Split "New York, NY" into city = "New York" and state_abbr = "NY".
# n=1 splits only at the first comma, so it always produces exactly two columns
networks_df[['city', 'state_abbr']] = networks_df['location'].str.split(', ', n=1, expand=True)

# Quick check of how much the U.S. filter removed
print(len(all_networks), 'networks worldwide ->', len(networks_df), 'in the U.S.')

815 networks worldwide -> 60 in the U.S.


## 3. CityBikes: count stations and docks per network

We download every station in each U.S. network, waiting 1 second between networks so we don't overload the free API. This takes about a minute.

**Docks per station (`capacity`):** use the real dock count (`extra.slots`) if the station reports one, otherwise `free_bikes + empty_slots`.
Dockless systems (no `empty_slots`, or Lime's `999999` placeholder) get `NaN`, **not 0**.

Each network's results:
- `n_stations`: number of stations
- `total_docks`: total of `capacity` across its stations
- `pct_missing_capacity`: share of stations with no dock count (0–1). A value of 1.0 means the network is dockless, so its `total_docks` is blank.

In [19]:
# STEP 2: Download every station for each U.S. network

def fetch_stations(network_id):
    """Download all stations for one network; return [] on failure so one bad network doesn't stop the loop."""
    try:
        # Each network has its own endpoint that lists all of its stations
        response = requests.get(f'https://api.citybik.es/v2/networks/{network_id}', timeout=30)

        if response.status_code == 200:                    # 200 = request succeeded
            stations = response.json()['network']['stations']   # the station list sits inside 'network'
            for s in stations:
                s['network_id'] = network_id               # tag each station with its network so we can group later
            return stations

        # Any other status code (404, 500, etc.): report it and fall through to return []
        print(network_id, 'failed:', response.status_code)

    except requests.RequestException as e:
        # Covers timeouts, dropped connections, and other network errors
        print(network_id, 'error:', e)

    return []                                              # empty list = this network contributes no stations


# Loop over every U.S. network from networks_df and collect all stations into one big list
all_stations = []
for net_id in networks_df['network_id']:
    all_stations.extend(fetch_stations(net_id))            # extend adds each station individually, not as a nested list
    time.sleep(1)                                          # pause 1 second between calls so we don't overload the API



#STEP 3: Turn the raw station list into a table and estimate capacity

# json_normalize flattens nested fields, so the 'slots' value inside 'extra' becomes a column named 'extra.slots'.
# reindex keeps only the columns we need, and adds any column no station reported (e.g. extra.slots) as all-NaN instead of crashing
stations = pd.json_normalize(all_stations).reindex(
    columns=['network_id', 'id', 'free_bikes', 'empty_slots', 'extra.slots'])

# Some dockless systems (e.g. Lime) report a huge number like 999999 to mean "no limit."
# mask() turns those placeholders into NaN so they aren't counted as real docks
empty_slots = stations['empty_slots'].mask(stations['empty_slots'] >= 999999)

# Capacity = the station's true dock count (extra.slots) when the network reports it;
# otherwise estimate it as bikes currently parked + empty docks.
# If empty_slots is missing, the sum is NaN, which correctly marks the station as "capacity unknown"
stations['capacity'] = stations['extra.slots'].fillna(stations['free_bikes'] + empty_slots)

# Sanity check: no real bike station has 1,000+ docks, so if one does, a placeholder value slipped through
assert not (stations['capacity'] > 1000).any(), 'a station reports 1000+ docks - probably a placeholder value'



# STEP 4: Summarize stations into one row per network

citybike_df = (stations.groupby('network_id')              # one group per bike-share network
    .agg(
        n_stations=('id', 'count'),                        # number of stations in the network
        total_docks=('capacity', lambda c: c.sum(min_count=1)),           # total docks; all-NaN -> NaN, not 0
        pct_missing_capacity=('capacity', lambda c: c.isna().mean().round(4)))  # share of stations with unknown capacity (0 to 1)
    .reset_index()                                         # turn network_id back into a regular column
    # attach each network's name, city, and state from networks_df
    .merge(networks_df[['network_id', 'name', 'city', 'state_abbr']], on='network_id')
    # put the columns in a readable order
    [['network_id', 'name', 'city', 'state_abbr', 'n_stations', 'total_docks', 'pct_missing_capacity']]
    .sort_values('n_stations', ascending=False)            # largest networks first
    .reset_index(drop=True))                               # clean 0, 1, 2... index after sorting

citybike_df.head(10)                                       # preview the 10 largest networks

,network_id,name,city,state_abbr,n_stations,total_docks,pct_missing_capacity
0,citi-bike-nyc,Citi Bike,New York,NY,2466,73761.0,0.0
1,divvy,Divvy,Chicago,IL,2074,22412.0,0.0
2,capital-bikeshare,Capital BikeShare,Washington,DC,873,14795.0,0.0
3,bay-wheels,Bay Wheels,San Francisco Bay Area,CA,655,14310.0,0.0
4,blue-bikes,Blue Bikes,Boston,MA,624,10949.0,0.0
5,indego,Indego,Philadelphia,PA,322,5427.0,0.0
6,biketown,BIKETOWN,Portland,OR,308,3469.0,0.0
7,metro-bike-share,Metro Bike Share,Los Angeles,CA,223,3270.0,0.0
8,bublr-bikes,Bublr Bikes,Milwaukee,WI,173,1735.0,0.0
9,peacehealth-rides,PeaceHealth Rides,Eugene,OR,167,1644.0,0.0


Cleaning up any missing name values for Teammate

In [20]:
networks_df[networks_df['state_abbr'].isna()]
# These 5 U.S. networks list only a city with no ", ST" suffix, so the split above left state_abbr empty.
# Fill them in by hand, keyed on network_id so the fix still works if names or order change
missing_states = {
    'gobiki': 'HI',                 # Honolulu
    'greenbikeslc': 'UT',           # Salt Lake City
    'hibike': 'HI',                 # Kailua-Kona
    'joco-new-york': 'NY',          # New York City
    'wolf-ride-bike-share': 'NY',   # Stony Brook University, Long Island
}
networks_df['state_abbr'] = networks_df['state_abbr'].fillna(
    networks_df['network_id'].map(missing_states))

# Make sure no network is still missing a state
assert networks_df['state_abbr'].notna().all(), 'a network still has no state'

## 4. Census ACS 5-year (2024): every U.S. city

One request returns every city/town ("place") in every state. Variable codes come from
https://api.census.gov/data/2024/acs/acs5/variables.html.

If the request fails we only print the type of error. The `requests` error message contains the full URL, and the URL contains the key.

In [21]:
# Census variable codes -> readable column names we'll rename to later.
# The codes come from ACS tables: B01003 (population), B19013 (income),
# B08301 (means of transportation to work), B25044 (vehicles available by owner/renter)
CENSUS_VARS = {
    'B01003_001E': 'population',          # total population
    'B19013_001E': 'median_income',       # median household income ($)
    'B08301_001E': 'commuters_total',     # workers 16+
    'B08301_018E': 'commuters_bike',      # workers who bike to work
    'B25044_001E': 'households_total',    # occupied housing units
    'B25044_003E': 'no_vehicle_owner',    # owner households with no vehicle
    'B25044_010E': 'no_vehicle_renter',   # renter households with no vehicle
}

# Query parameters for the API call.
# Passing them as a dict (instead of building the URL by hand) lets requests handle the formatting,
# and keeps the key out of any URL string we might accidentally print
params_census = {
    'get': 'NAME,' + ','.join(CENSUS_VARS),   # joining a dict joins its keys: "NAME,B01003_001E,B19013_001E,..."
    'for': 'place:*',                         # every city/town...
    'in': 'state:*',                          # ...in every state
    'key': CENSUS_API_KEY,                    # loaded from Colab Secrets or .env, never typed here
}

try:
    # 2024 = the 2020-2024 ACS 5-year estimates. Timeout is longer than CityBikes
    # because this one call returns ~30,000 places
    response = requests.get('https://api.census.gov/data/2024/acs/acs5', params=params_census, timeout=60)
except requests.RequestException as e:
    # Connection errors often include the full request URL, key included, in their message.
    # So we report only the error type, and 'from None' hides the original traceback that would show it
    raise RuntimeError(f'Census request failed ({type(e).__name__})') from None

if response.status_code != 200:
    # Show the status code and the start of the Census error message (e.g. "unknown variable")
    # so we can debug, without printing response.url, which contains the key
    raise RuntimeError(f'Census API returned {response.status_code}: {response.text[:300]}')

census_raw = response.json()          # list of rows; the first row is the header

## 5. Clean the Census data

- **geoid** = state code + place code. This is the unique ID for each city.
- Numbers arrive as text, so we convert them. **Negative values (like `-666666666`) are Census codes for "no data", so they become `NaN`.**
- `"Chicago city, Illinois"` becomes `city = "Chicago"` and `state = "Illinois"`.
- `pct_bike_commute = commuters_bike / commuters_total`
- `pct_no_vehicle = (no_vehicle_owner + no_vehicle_renter) / households_total`

In [22]:
# census_raw is a list of lists: row 0 is the header, the rest are data rows.
# Build a DataFrame from the data rows using the header as column names,
# Rename variable codes to readable names
raw = pd.DataFrame(census_raw[1:], columns=census_raw[0]).rename(columns=CENSUS_VARS)

# The API returns every value as text, so convert our numeric columns to numbers.
# errors='coerce' turns anything that can't be converted into NaN instead of crashing
num_cols = list(CENSUS_VARS.values())
raw[num_cols] = raw[num_cols].apply(pd.to_numeric, errors='coerce')

# The Census uses large negative numbers (e.g. -666666666) to mean "no data / too few responses".
# No real count or income is negative, so mask() replaces every negative value with NaN
raw[num_cols] = raw[num_cols].mask(raw[num_cols] < 0)

# GEOID = 2-digit state code + 5-digit place code (e.g. "36" + "51000" = NYC).
# This is the unique ID we'll join on, since city names are not unique across states
census_df = pd.DataFrame({'geoid': raw['state'] + raw['place']})

# NAME looks like "New York city, New York". rsplit at the LAST comma
# separates the place name from the state name
census_df[['city', 'state']] = raw['NAME'].str.rsplit(', ', n=1, expand=True)

# Remove the Census place-type suffix so names look like CityBikes' ("New York city" -> "New York").
# shortcut: strips only common place types; odd names like "Nashville-Davidson metropolitan government (balance)" stay as-is
census_df['city'] = census_df['city'].str.replace(r'\s+(city|town|village|borough|CDP|municipality)$', '', regex=True)

# Copy over the raw measures we'll analyze directly
census_df['population'] = raw['population']
census_df['median_income'] = raw['median_income']

# Turn counts into shares (0 to 1) so cities of different sizes can be compared:
# share of workers who bike to work
census_df['pct_bike_commute'] = (raw['commuters_bike'] / raw['commuters_total']).round(4)
# share of households (owners + renters) with no vehicle
census_df['pct_no_vehicle'] = ((raw['no_vehicle_owner'] + raw['no_vehicle_renter']) / raw['households_total']).round(4)

# Sanity check: a share must be between 0 and 1.
# stack() lines both columns up into one series, and dropna() skips places with missing data
pct = census_df[['pct_bike_commute', 'pct_no_vehicle']]
assert pct.stack().dropna().between(0, 1).all(), 'a share is outside 0-1'

#Largest places first, with a clean 0,1,2... index
census_df = census_df.sort_values('population', ascending=False).reset_index(drop=True)
census_df.head(10)       # preview the 10 most populous places

,geoid,city,state,population,median_income,pct_bike_commute,pct_no_vehicle
0,3651000,New York,New York,8483844,80483.0,0.0159,0.5533
1,0644000,Los Angeles,California,3857263,81939.0,0.0059,0.1188
2,1714000,Chicago,Illinois,2711226,77902.0,0.0142,0.2659
3,4835000,Houston,Texas,2328253,64813.0,0.0048,0.1006
4,0455000,Phoenix,Arizona,1642323,81332.0,0.0040,0.0660
5,4260000,Philadelphia,Pennsylvania,1579706,61953.0,0.0184,0.2791
6,4865000,San Antonio,Texas,1479835,65056.0,0.0025,0.0769
7,0666000,San Diego,California,1389526,108077.0,0.0067,0.0660
8,4819000,Dallas,Texas,1307930,70518.0,0.0015,0.0881
9,0668000,San Jose,California,990138,146427.0,0.0051,0.0614


## 6. Save the two CSV files

In [25]:
citybike_df.to_csv(DATA_DIR / 'citybike_networks_us.csv', index=False)
census_df.to_csv(DATA_DIR / 'census_places.csv', index=False)
print(f'citybike_networks_us.csv: {len(citybike_df):,} rows')
print(f'census_places.csv: {len(census_df):,} rows')

citybike_networks_us.csv: 56 rows
census_places.csv: 32,330 rows


7. Use the two CSV files as a data frame so the group can use.

In [24]:
import pandas as pd
# Make into data frames so group can just take the data and work from there

census = pd.read_csv('data/raw/census_places.csv', dtype={'geoid': str})
bikes = pd.read_csv('data/raw/citybike_networks_us.csv')
#Printing Data to see how it looks for team
display(census.head())
print("""
        """)
display(bikes.head())


,geoid,city,state,population,median_income,pct_bike_commute,pct_no_vehicle
0,3651000,New York,New York,8483844,80483.0,0.0159,0.5533
1,0644000,Los Angeles,California,3857263,81939.0,0.0059,0.1188
2,1714000,Chicago,Illinois,2711226,77902.0,0.0142,0.2659
3,4835000,Houston,Texas,2328253,64813.0,0.0048,0.1006
4,0455000,Phoenix,Arizona,1642323,81332.0,0.0040,0.0660


,network_id,name,city,state_abbr,n_stations,total_docks,pct_missing_capacity
0,citi-bike-nyc,Citi Bike,New York,NY,2466,73761.0,0.0
1,divvy,Divvy,Chicago,IL,2074,22412.0,0.0
2,capital-bikeshare,Capital BikeShare,Washington,DC,873,14795.0,0.0
3,bay-wheels,Bay Wheels,San Francisco Bay Area,CA,655,14310.0,0.0
4,blue-bikes,Blue Bikes,Boston,MA,624,10949.0,0.0
